# US-07 - Regresión logística multinomial (M1): baseline y mejora

**Objetivo y alcance.** Entrenar la regresión logística multinomial (softmax) sobre el Pipeline compartido del proyecto (US-05), primero como *baseline* y después mejorada, para obtener una configuración final comparable con los otros tres modelos en US-11.

**Reglas del proyecto aplicadas:**
- Entrenamiento **solo** con `data/train_70.csv`; `val_15.csv` solo para evaluar. `test_15.csv` **no se carga ni se usa** (D-012).
- Imputación (D-011, aprobada), encoding (D-014, aprobada) y escalado viven **dentro** del Pipeline; nada se aplica a mano a los datos (D-013, ítem 3).
- Semilla única importada de `src.config.RANDOM_STATE`; no se crean semillas nuevas.
- Métrica principal: **macro-F1** en validación (D-010). Límite de sobreajuste: `accuracy(train) - accuracy(val) < 5` puntos porcentuales, en absoluto; exactamente 5.00 **no** cumple (D-010, respuesta del profesor).
- Toda cifra reportada debe venir de una ejecución registrada (AGENTS.md §5): las salidas de este notebook provienen de ejecuciones reales y las métricas quedan registradas en `results/logistic_regression.csv` (nada inventado).

**Estado de dependencias:** D-011 (imputación, aprobada por Brian el 2026-11-13) y **D-014 (encoding, aprobada por Brian el 2026-11-16 vía PR #61, ítem 2 corregido)** están conectadas por parámetros a través de `build_preprocessing()` (D-014, ítem 5: el módulo compartido no se modifica). El `StandardScaler` es paso propio de M1 (D-014, ítem 6). **US-05 fase 2** (los defaults del módulo compartido) sigue sin integrarse en `develop`, así que **todas las métricas de este notebook se marcan PROVISIONALES** hasta que aterrice y se re-ejecute (D-013, Consequences).

## 1. Importaciones y configuración

Importamos la semilla del proyecto desde `src.config` y el modelo reutilizable de `src.model_logreg`, que a su vez usa el preprocesamiento compartido de `src.preprocessing` (`build_preprocessing`). No duplicamos ninguna lógica de `src/`.

In [1]:
import sys
from pathlib import Path

import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, classification_report, f1_score

from src.config import RANDOM_STATE
from src.model_logreg import CANDIDATE_CONFIGS, build_logreg_pipeline
from src.preprocessing import make_features_target

print(f"Python: {sys.version.split()[0]} | pandas: {pd.__version__}")
print(f"RANDOM_STATE (src.config): {RANDOM_STATE}")

Python: 3.12.10 | pandas: 3.1.1
RANDOM_STATE (src.config): 42


## 2. Carga de datos

Cargamos **únicamente** las particiones compartidas permitidas para entrenar y validar. `test_15.csv` se reserva para la evaluación final (US-11) y no aparece en este notebook. Ejecutar desde la raíz del repositorio.

In [2]:
DATA = Path("data")
train_df = pd.read_csv(DATA / "train_70.csv")
val_df = pd.read_csv(DATA / "val_15.csv")

print(f"train_70: {train_df.shape} | val_15: {val_df.shape}")
print(train_df.head().to_string())

train_70: (5647, 11) | val_15: (1210, 11)
       ID  Gender Ever_Married  Age Graduated Profession  Work_Experience Spending_Score  Family_Size  Var_1 Segmentation
0  460968    Male          Yes   30        No     Doctor             11.0        Average          2.0  Cat_3            A
1  461619    Male          Yes   53       Yes  Executive              NaN           High          3.0  Cat_6            C
2  467217  Female          Yes   79       Yes     Lawyer              NaN           High          2.0  Cat_6            C
3  459338    Male          Yes   88        No  Executive              NaN            Low          NaN  Cat_6            D
4  465826    Male          Yes   45        No  Executive              3.0            Low          9.0  Cat_4            D


## 3. Separación de variables predictoras y target

`make_features_target` (US-05) selecciona las features por nombre desde `src.config`, excluye `ID` (nunca es feature) y devuelve el target aparte, que nunca entra en el preprocesamiento. El dataframe de entrada no se modifica.

In [3]:
X_train, y_train = make_features_target(train_df)
X_val, y_val = make_features_target(val_df)

print(f"X_train: {X_train.shape} | X_val: {X_val.shape}")
print(f"Features ({len(X_train.columns)}): {list(X_train.columns)}")
print(f"Clases del target: {sorted(y_train.unique())}")

X_train: (5647, 9) | X_val: (1210, 9)
Features (9): ['Age', 'Work_Experience', 'Family_Size', 'Ever_Married', 'Graduated', 'Gender', 'Profession', 'Var_1', 'Spending_Score']
Clases del target: ['A', 'B', 'C', 'D']


## 4. Baseline: DummyClassifier

Referencia mínima de D-010: `DummyClassifier(strategy="most_frequent")` predice siempre la clase mayoritaria (D). Todo modelo real debe superar claramente su macro-F1 en validación. Esta celda **no depende de D-014** y puede ejecutarse ya: el baseline no necesita encoding porque ignora las features.

In [4]:
baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)

for name, X, y in (("train_70", X_train, y_train), ("val_15", X_val, y_val)):
    pred = baseline.predict(X)
    print(
        f"{name}: accuracy={accuracy_score(y, pred):.4f} | "
        f"macro_f1={f1_score(y, pred, average='macro'):.4f}"
    )

train_70: accuracy=0.2810 | macro_f1=0.1097
val_15: accuracy=0.2810 | macro_f1=0.1097


## 5. Pipeline de Logistic Regression (estructura)

El pipeline de M1 concatena: (1) el preprocesamiento compartido con los imputers aprobados de D-011 y los encoders aprobados de D-014 (conectados por parámetros desde `src.model_logreg`), (2) el `StandardScaler` específico de regresión logística (D-014, ítem 6: después del preprocesamiento compartido, antes del estimador) y (3) `LogisticRegression` multinomial (por defecto en scikit-learn 1.9). La celda marca `D014_APPROVED = True` (D-014 aprobada por Brian el 2026-11-16, PR #61).

In [5]:
from src import preprocessing

print("Slots del módulo compartido (US-05 sigue en fase 1; este modelo los conecta por parámetros):")
print(preprocessing.PENDING_DECISIONS)

# D-014 aprobada por Brian el 2026-11-16 (PR #61): los encoders se conectan
# por parámetros desde src.model_logreg (D-014, ítem 5); el módulo compartido
# no se modifica. US-05 fase 2 no está integrada aún -> métricas PROVISIONALES.
D014_APPROVED = True

model = build_logreg_pipeline()  # baseline: C=1.0, penalty=l2, solver=lbfgs
print(f"\nD014_APPROVED = {D014_APPROVED}")
print(f"Pasos del Pipeline: {[s[0] for s in model.steps]}")

Slots del módulo compartido (US-05 sigue en fase 1; este modelo los conecta por parámetros):
{'imputation': 'D-011 (Approved by Brian, 2026-11-13) — strategies approved, implementation deferred to phase 2', 'encoding': 'not yet recorded; candidate identifier D-014, pending team approval'}

D014_APPROVED = True
Pasos del Pipeline: ['preprocessing', 'scaler', 'classifier']


## 6. Entrenamiento y evaluación

Se entrena **solo con `train_70`** y se evalúa en `val_15` con accuracy, macro-F1 y el reporte por clase. La imputación, el encoding y el escalado se ajustan exclusivamente con datos de entrenamiento; `val_15` solo se transforma (sin fuga de datos). Tras el ajuste se comprueba el ancho real de la matriz transformada. Mientras US-05 fase 2 no se integre en `develop`, las métricas quedan marcadas como **PROVISIONALES**.

In [6]:
model.fit(X_train, y_train)

width = model.named_steps["preprocessing"].transform(X_train).shape[1]
print(f"Ancho de la matriz tras el preprocesamiento (ajustado en train_70): {width} columnas")

for name, X, y in (("train_70", X_train, y_train), ("val_15", X_val, y_val)):
    pred = model.predict(X)
    print(
        f"--- {name} --- accuracy={accuracy_score(y, pred):.4f} | "
        f"macro_f1={f1_score(y, pred, average='macro'):.4f}"
    )

print(classification_report(y_val, model.predict(X_val), digits=4))

Ancho de la matriz tras el preprocesamiento (ajustado en train_70): 28 columnas
--- train_70 --- accuracy=0.5128 | macro_f1=0.4894
--- val_15 --- accuracy=0.5058 | macro_f1=0.4797
              precision    recall  f1-score   support

           A     0.4007    0.4088    0.4047       296
           B     0.3669    0.2222    0.2768       279
           C     0.4957    0.5797    0.5344       295
           D     0.6548    0.7588    0.7030       340

    accuracy                         0.5058      1210
   macro avg     0.4795    0.4924    0.4797      1210
weighted avg     0.4874    0.5058    0.4906      1210



C:\Users\lorig\customer-segmentation-multiclass\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


## 7. Análisis de sobreajuste (D-010)

El criterio oficial compara accuracy de train y validación como **diferencia absoluta** en puntos porcentuales: cumple si es **estrictamente menor que 5 pp**; exactamente 5.00 no cumple. El gap de macro-F1 se reporta como informativo.

In [7]:
acc_train = accuracy_score(y_train, model.predict(X_train))
acc_val = accuracy_score(y_val, model.predict(X_val))
gap_pp = (acc_train - acc_val) * 100

print(f"accuracy(train_70) = {acc_train:.4f}")
print(f"accuracy(val_15)   = {acc_val:.4f}")
print(f"diferencia         = {gap_pp:.2f} pp -> "
      f"{'CUMPLE' if gap_pp < 5.0 else 'NO CUMPLE'} el límite de D-010 (< 5 pp)")

accuracy(train_70) = 0.5128
accuracy(val_15)   = 0.5058
diferencia         = 0.71 pp -> CUMPLE el límite de D-010 (< 5 pp)


## 8. Comparación de experimentos (mejora)

Las configuraciones candidatas (`CANDIDATE_CONFIGS` en `src/model_logreg.py`: fuerza de regularización `C`, `l1`/`l2` con `saga`, `class_weight`) se comparan con **validación cruzada estratificada de 5 particiones sobre `train_70`**, pasando el **Pipeline completo** a `cross_validate` (D-010, ítem 6): así la imputación, el encoding y el escalado se ajustan dentro de cada fold y no hay fuga de información. La comparación oficial final sigue siendo train vs `val_15`; todas las cifras son **PROVISIONALES** hasta que US-05 fase 2 se integre en `develop`.

In [8]:
from sklearn.model_selection import StratifiedKFold, cross_validate

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = {"accuracy": "accuracy", "macro_f1": "f1_macro"}

rows = []
for cfg in CANDIDATE_CONFIGS:
    pipe = build_logreg_pipeline(
        C=cfg["C"],
        penalty=cfg["penalty"],
        solver=cfg["solver"],
        class_weight=cfg["class_weight"],
    )
    scores = cross_validate(
        pipe, X_train, y_train, cv=cv, scoring=scoring, return_train_score=True
    )
    acc, f1 = scores["test_accuracy"], scores["test_macro_f1"]
    rows.append(
        {
            "config": cfg["config"],
            "cv_accuracy_mean": acc.mean(),
            "cv_accuracy_std": acc.std(),
            "cv_macro_f1_mean": f1.mean(),
            "cv_macro_f1_std": f1.std(),
            "train_cv_acc_gap_pp": (scores["train_accuracy"].mean() - acc.mean()) * 100,
        }
    )

print(pd.DataFrame(rows).sort_values("cv_macro_f1_mean", ascending=False).to_string(index=False))

         config  cv_accuracy_mean  cv_accuracy_std  cv_macro_f1_mean  cv_macro_f1_std  train_cv_acc_gap_pp
C=1_l2_balanced          0.499024         0.009642          0.485673         0.010076             1.394710
       C=0.1_l2          0.501857         0.014946          0.479499         0.017332             1.377112
    C=1_l1_saga          0.501503         0.014235          0.479009         0.016764             1.416939
        C=10_l2          0.501326         0.012772          0.478991         0.015005             1.461180
       baseline          0.501149         0.012641          0.478885         0.014929             1.492177
      C=0.01_l2          0.500794         0.017039          0.477198         0.019487             1.487833


C:\Users\lorig\customer-segmentation-multiclass\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
C:\Users\lorig\customer-segmentation-multiclass\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
C:

## 9. Registro de resultados (convención US-03)

Cada ejecución real se añade (append-only) a `results/logistic_regression.csv` con las columnas de la convención: `run_timestamp_utc, seed, model, config, partition, accuracy, macro_f1, f1_A..f1_D, notes`. Nunca se editan ni borran filas anteriores. Mientras US-05 fase 2 no esté integrada en `develop`, **todas las filas llevan la marca PROVISIONAL en `notes`**: no son comparables de forma definitiva con los otros modelos.

Se registran: el baseline del `DummyClassifier` (train/val), y para cada configuración candidata (1) accuracy/macro-F1 con ajuste completo en `train_70` (partición `train_70`), (2) su evaluación en `val_15` con el gap de sobreajuste D-010 en `notes` y (3) sus predicciones out-of-fold de una CV estratificada de 5 folds sobre `train_70` (partición `cv5fold`).

In [9]:
from datetime import datetime, timezone

from sklearn.model_selection import StratifiedKFold, cross_val_predict

RESULTS_CSV = Path("results") / "logistic_regression.csv"
PROVISIONAL = "PROVISIONAL (US-05 fase 2 no integrada en develop)"

def record(config: str, partition: str, y_true, y_pred, notes: str = "") -> None:
    """Añade una fila de métricas de una ejecución real (append-only)."""
    row = pd.DataFrame(
        [
            {
                "run_timestamp_utc": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
                "seed": RANDOM_STATE,
                "model": "logistic_regression",
                "config": config,
                "partition": partition,
                "accuracy": accuracy_score(y_true, y_pred),
                "macro_f1": f1_score(y_true, y_pred, average="macro"),
                "f1_A": f1_score(y_true, y_pred, average=None, labels=["A"])[0],
                "f1_B": f1_score(y_true, y_pred, average=None, labels=["B"])[0],
                "f1_C": f1_score(y_true, y_pred, average=None, labels=["C"])[0],
                "f1_D": f1_score(y_true, y_pred, average=None, labels=["D"])[0],
                "notes": notes,
            }
        ]
    )
    row.to_csv(RESULTS_CSV, mode="a", header=not RESULTS_CSV.exists(), index=False)

# --- 1) Ejecuciones reales: ajuste completo en train_70, evaluación en val_15
#          y predicciones out-of-fold de la CV-5 (mismos folds que la sección 8).
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
preds_train, preds_val, oof_preds, summary = {}, {}, {}, []
for cfg in CANDIDATE_CONFIGS:
    pipe = build_logreg_pipeline(
        C=cfg["C"],
        penalty=cfg["penalty"],
        solver=cfg["solver"],
        class_weight=cfg["class_weight"],
    )
    pipe.fit(X_train, y_train)
    preds_train[cfg["config"]] = pipe.predict(X_train)
    preds_val[cfg["config"]] = pipe.predict(X_val)
    oof_preds[cfg["config"]] = cross_val_predict(pipe, X_train, y_train, cv=cv)
    gap = (accuracy_score(y_train, preds_train[cfg["config"]])
           - accuracy_score(y_val, preds_val[cfg["config"]])) * 100
    summary.append(
        {
            "config": cfg["config"],
            "acc_train": accuracy_score(y_train, preds_train[cfg["config"]]),
            "acc_val": accuracy_score(y_val, preds_val[cfg["config"]]),
            "macro_f1_val": f1_score(y_val, preds_val[cfg["config"]], average="macro"),
            "macro_f1_cv": f1_score(y_train, oof_preds[cfg["config"]], average="macro"),
            "gap_pp": gap,
            "cumple_D010": gap < 5.0,
        }
    )

# --- 2) Configuración provisional elegida: macro-F1 en val_15 entre las que
#          cumplen el límite de sobreajuste de D-010 (< 5 pp).
valid = [s for s in summary if s["cumple_D010"]]
elegida = max(valid, key=lambda s: s["macro_f1_val"]) if valid else None
if elegida is None:
    print("Ninguna configuración cumple el límite de D-010; no se marca candidato final.")

# --- 3) Registro append-only (incluye el baseline del DummyClassifier).
record("baseline_dummy", "train_70", y_train, baseline.predict(X_train), PROVISIONAL)
record(
    "baseline_dummy", "val_15", y_val, baseline.predict(X_val),
    PROVISIONAL + " | referencia mínima D-010",
)
for s in summary:
    extra = (
        f" | gap D-010 {s['gap_pp']:+.2f} pp "
        + ("CUMPLE" if s["cumple_D010"] else "NO CUMPLE")
    )
    if elegida is not None and s["config"] == elegida["config"]:
        extra += " | CANDIDATO FINAL provisional para US-11"
    record(s["config"], "train_70", y_train, preds_train[s["config"]], PROVISIONAL)
    record(s["config"], "val_15", y_val, preds_val[s["config"]], PROVISIONAL + extra)
    record(
        s["config"], "cv5fold", y_train, oof_preds[s["config"]],
        PROVISIONAL + " | OOF StratifiedKFold(5, seed src.config)",
    )

print(f"\nFilas añadidas a {RESULTS_CSV}: {2 + 3 * len(summary)} (todas PROVISIONALES)")
print(pd.DataFrame(summary).sort_values("macro_f1_val", ascending=False).to_string(index=False))


Filas añadidas a results\logistic_regression.csv: 20 (todas PROVISIONALES)
         config  acc_train  acc_val  macro_f1_val  macro_f1_cv   gap_pp  cumple_D010
       C=0.1_l2   0.512130 0.506612      0.480616     0.479686 0.551876         True
    C=1_l1_saga   0.513901 0.506612      0.480436     0.479209 0.728962         True
       baseline   0.512839 0.505785      0.479710     0.479067 0.705355         True
        C=10_l2   0.512485 0.504959      0.478983     0.479162 0.752583         True
C=1_l2_balanced   0.512307 0.495868      0.478583     0.485698 1.643965         True
      C=0.01_l2   0.513901 0.502479      0.475741     0.477401 1.142185         True


C:\Users\lorig\customer-segmentation-multiclass\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
C:\Users\lorig\customer-segmentation-multiclass\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
C:

## 10. Análisis de resultados y conclusiones

**Ejecutado el 2026-11-16.** Todas las cifras provienen de ejecuciones reales registradas en `results/logistic_regression.csv` (20 filas: 2 del Dummy + 3 por cada configuración candidata; **todas marcadas PROVISIONALES** porque US-05 fase 2 no está integrada en `develop`). Semilla `RANDOM_STATE = 42` en todas las filas.

**Baseline (referencia mínima D-010).** `DummyClassifier(strategy="most_frequent")`: en `val_15` accuracy 0.2810 / macro-F1 0.1097 (predice siempre D; f1_D ≈ 0.4387).

**Dimensiones.** El preprocesamiento con D-011 + D-014 conectados por parámetros produce **28 columnas** ajustado en `train_70` (3 numéricas + 2 indicadores de nulos + 2 binarias + 1 `Gender` + 10 `Profession` con `Unknown` + 7 `Var_1` + 3 `Spending_Score`) — coincide con el objetivo de D-014.

**Resultados por configuración (ajuste completo en `train_70`, evaluación en `val_15`):**

| config | acc val | macro-F1 val | gap D-010 | macro-F1 CV OOF |
|---|---:|---:|---:|---:|
| C=0.1_l2 | 0.5066 | **0.4806** | +0.55 pp | 0.4797 |
| C=1_l1_saga | 0.5066 | 0.4804 | +0.73 pp | 0.4792 |
| baseline (C=1_l2) | 0.5058 | 0.4797 | +0.71 pp | 0.4791 |
| C=10_l2 | 0.5050 | 0.4790 | +0.75 pp | 0.4792 |
| C=1_l2_balanced | 0.4959 | 0.4786 | +1.64 pp | 0.4857 |
| C=0.01_l2 | 0.5025 | 0.4757 | +1.14 pp | 0.4774 |

- **Todas** las configuraciones cumplen el límite de sobreajuste de D-010 (gap < 5 pp; el máximo es +1.64 pp) y superan con claridad al Dummy (0.1097).
- **Configuración provisional elegida: `C=0.1_l2`** (`C=0.1, penalty=l2, solver=lbfgs, max_iter=1000`): mayor macro-F1 en `val_15` (0.4806) entre las que cumplen D-010. La CV-5 OOF sobre `train_70` (sección 8) la sitúa también primera en macro-F1 medio (0.4797) con desviaciones bajas (~0.01–0.02).
- `C=1_l2_balanced` lidera la CV por macro-F1 (0.4857) porque recupera la clase B (f1_B 0.30 OOF), pero su macro-F1 en `val_15` (0.4786) y su accuracy (0.4959) son menores; la métrica oficial de D-010 en validación mantiene `C=0.1_l2` como candidato.
- La clase B es la más difícil (f1 ≈ 0.27–0.30 en val en todas las configs); A ≈ 0.40–0.41, C ≈ 0.53–0.54, D ≈ 0.69–0.70.

**Limitaciones pendientes:**
1. **PROVISIONAL:** US-05 fase 2 no está integrada en `develop`. Este notebook ya conecta D-011/D-014 por parámetros (mecanismo de D-014, ítem 5) y mide 28 columnas, pero las métricas deben re-ejecutarse cuando la fase 2 aterrice (D-013, Consequences) antes de la comparación final de US-11.
2. scikit-learn 1.9.1 marca `penalty` como deprecado (se elimina en 1.10; migración futura a `l1_ratio`). Con el entorno fijado en `requirements.txt` los resultados son reproducibles (verificado: re-ejecución idéntica a las filas registradas).
3. La comparación definitiva con los otros modelos ocurre en US-11 sobre la partición compartida (D-013); estos números no son comparables de forma definitiva hasta entonces.
4. `test_15.csv` no se carga ni se usa en ningún punto (D-012); se abrirá una sola vez en US-12 con el modelo congelado.